In [1]:
!du -hs Emilia-YODAS_permutate

423G	Emilia-YODAS_permutate


In [2]:
# !wget https://gist.githubusercontent.com/huseinzol05/98974ae8c6c7a65d4bc0af9f5003786a/raw/2e06e71ef7349a57bc58cc9913ae6bae1f9f8447/mp.py

In [3]:
from glob import glob
import os

repository = 'malaysia-ai/Emilia-YODAS-Voice-Conversion'
folder = 'Emilia-YODAS_permutate'
files = glob(f'{folder}/**/*.json', recursive = True)
len(files)

331717

In [4]:
import zipfile
import time
from huggingface_hub import HfFileSystem
from huggingface_hub import HfApi
from tqdm import tqdm
from multiprocess import Pool
import itertools

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))
        
api = HfApi()
partition_size = 5e+9

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
def loop(files):
    files, index = files
    current_index = 0
    api = HfApi()
    fs = HfFileSystem()
    total = 0
    temp = []
    for i in tqdm(range(len(files))):
        s = os.stat(files[i]).st_size
        if s + total >= partition_size:
            part_name = f"{folder}-{index}-{current_index}.zip"
                
            with zipfile.ZipFile(part_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
                for f in temp:
                    zipf.write(f, arcname=f)

            while True:
                try:
                    api.upload_file(
                        path_or_fileobj=part_name,
                        path_in_repo=part_name,
                        repo_id=repository,
                        repo_type="dataset",
                    )
                    break
                except:
                    time.sleep(60)

            os.remove(part_name)
            
            current_index += 1
            temp = [files[i]]
            total = s
        else:
            temp.append(files[i])
            total += s
        
    if len(temp):
        part_name = f"{folder}-{index}-{current_index}.zip"

        with zipfile.ZipFile(part_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for f in temp:
                zipf.write(f, arcname=f)

        while True:
            try:
                api.upload_file(
                    path_or_fileobj=part_name,
                    path_in_repo=part_name,
                    repo_id=repository,
                    repo_type="dataset",
                )
                break
            except:
                time.sleep(60)

        os.remove(part_name)

In [6]:
loop((files[:1000], 0))

100%|██████████| 1000/1000 [00:00<00:00, 449357.62it/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████|  113MB /  113MB,  172MB/s  
Processing Files (1 / 1): 100%|██████████|  113MB /  113MB,  155MB/s  
New Data Upload: 100%|██████████|  197kB /  197kB,  489kB/s  


In [ ]:
multiprocessing(files, loop, cores = 10, returned = False)

Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████|  622MB /  622MB,  384MB/s  
Processing Files (1 / 1): 100%|██████████|  622MB /  622MB,  299MB/s  
New Data Upload: 100%|██████████|  410kB /  410kB,  228kB/s  
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████|  578MB /  578MB,  419MB/s  
Processing Files (1 / 1): 100%|██████████|  578MB /  578MB,  359MB/s  
New Data Upload: 100%|██████████|  490kB /  490kB,  350kB/s  
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████|  523MB /  523MB,  400MB/s  
Processing Files (1 / 1): 100%|██████████|  523MB /  523MB,  301MB/s  
New Data Upload: 100%|██████████|  591kB /  591kB,  369kB/s  
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████|  518MB /  518MB,  277MB/s  
Processing Files (1 / 1): 100%|██████████|  518MB